# 03b — LIME Explanations

Computes per-replica LIME attributions on a subsample of the flagged set F_{A,B}
for all three model types. Drops into notebook 04's drift code identically to SHAP,
just under `data/lime/{MODEL_TYPE}/pair_XX/`.

## Scope decisions

| Decision | Value | Why |
|---|---|---|
| Explainer | `lime.lime_tabular.LimeTabularExplainer` | Standard choice for tabular |
| Instances per pair | `N_LIME_SUBSAMPLE = 200` | Flagged set subsample; keeps runtime tractable |
| Perturbations per instance | `num_samples = 1000` | Paper default is 5000; 1000 is ~90% as stable at 5× speedup |
| Features returned | `num_features = p` (all) | Needed for full cosine + RBO over the complete attribution vector |
| Sampling basis | **Per-window training data, raw** | A replicas explained against `X[idx_A]`; B replicas against `X[idx_B]`. Matches the methodology's per-window explanation framing |
| Categorical declarations | The binary indicator columns | LIME samples these from the empirical distribution instead of as continuous |

## Per-model-type specifics

All callables receive **raw** feature input. Each replica's own pipeline / scaler
handles whatever scaling the model needs internally — LIME never pre-scales.

- **XGBoost**: trained on raw features; `predict_proba` consumes raw input directly.
- **LR**: each replica is a `Pipeline(StandardScaler, LogisticRegression)`; the
  pipeline scales internally before predicting.
- **MLP-PLR**: the `predict_proba` callable wraps the model with its replica
  scaler and pushes batched inputs to GPU. **Requires GPU runtime.**

## Output layout

```
data/lime/{MODEL_TYPE}/pair_{pid:02d}/
  lime_A.npy                 # (R, |F_lime|, p), float32 — attribution vectors
  lime_B.npy                 # same
  lime_flagged_idx.npy       # (|F_lime|,) int — subsample positions within flagged_idx
  stochasticity.json         # {best_replica_A, max_abs_diff, median_abs_diff, is_deterministic}
  global_importance_pair00_A.png   # top-20 plot for pair 0
```

## Runtime estimate (T4 GPU, Home Credit scale ~150-200 features)

- XGBoost:  ~20-30 min / pair
- LR:       ~10-15 min / pair
- MLP-PLR:  ~30-45 min / pair (GPU predict_proba callable)

## Stochasticity note

LIME is much more stochastic than both TreeSHAP (deterministic) and GradientExplainer
(mildly stochastic). `max_abs_diff` in stochasticity.json will be non-zero and typically
larger than SHAP's. This is a central thesis finding, not a bug.

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
%pip install -q lime

In [19]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

import lime
import lime.lime_tabular

WORKSPACE = Path('/content/drive/MyDrive/HomeCredit_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'

# ── Model type — must match the MODEL_TYPE used in notebook 02 / 02b ─────────
MODEL_TYPE = 'mlp_plr'   # 'xgboost' | 'logreg' | 'mlp_plr'

MODEL_DIR = WORKSPACE / 'data' / 'models' / MODEL_TYPE
LIME_DIR  = WORKSPACE / 'data' / 'lime'  / MODEL_TYPE
LIME_DIR.mkdir(parents=True, exist_ok=True)

# ── Runtime knobs (see header for rationale) ─────────────────────────────────
N_LIME_SUBSAMPLE = 200     # flagged instances sampled per pair for LIME (runtime cap)
NUM_SAMPLES      = 1000    # perturbations per LIME call (paper default 5000)

from importlib.metadata import version
print(f'lime version: {version("lime")}')
print(f'MODEL_TYPE : {MODEL_TYPE}')
print(f'N_LIME_SUBSAMPLE: {N_LIME_SUBSAMPLE}  NUM_SAMPLES: {NUM_SAMPLES}')

lime version: 0.2.0.1
MODEL_TYPE : mlp_plr
N_LIME_SUBSAMPLE: 200  NUM_SAMPLES: 1000


In [20]:
# Load data and window config
X = pd.read_parquet(PROC_DIR / 'X.parquet').values.astype(np.float32)

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names_json = json.load(f)
feature_names = feature_names_json['all']

num_col_idx = [feature_names.index(fn) for fn in feature_names_json['num']]
bin_col_idx = [i for i in range(len(feature_names)) if i not in set(num_col_idx)]
n_num = len(num_col_idx)
n_bin = len(bin_col_idx)

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R     = config['parameters']['R']
pairs = config['pairs']

print(f'X: {X.shape}, features: {len(feature_names)}')
print(f'R={R}, {len(pairs)} pairs, numeric={n_num}, binary={n_bin}')

X: (600000, 683), features: 683
R=2, 5 pairs, numeric=679, binary=4


In [21]:
# ═════════════════════════════════════════════════════════════════════════════
# MLP-PLR setup — only runs when MODEL_TYPE == 'mlp_plr'.
# Classes copied byte-identically from 02b/03.
# ═════════════════════════════════════════════════════════════════════════════

if MODEL_TYPE == 'mlp_plr':
    import math
    import torch
    import torch.nn as nn

    assert torch.cuda.is_available(), 'MLP-PLR LIME requires a GPU runtime.'
    DEVICE = torch.device('cuda')
    print(f'PyTorch : {torch.__version__}')
    print(f'Device  : {torch.cuda.get_device_name(0)}')


    class PLREmbedding(nn.Module):
        """Vectorised PLR embedding for all numeric features at once."""

        def __init__(self, n_num_features: int, k_periodic: int,
                     sigma_init: float, d_embedding: int):
            super().__init__()
            self.n_num_features = n_num_features
            self.k_periodic     = k_periodic
            self.d_embedding    = d_embedding
            self.c = nn.Parameter(torch.randn(n_num_features, k_periodic) * sigma_init)
            self.W = nn.Parameter(torch.empty(n_num_features, 2 * k_periodic, d_embedding))
            self.b = nn.Parameter(torch.zeros(n_num_features, d_embedding))
            nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))

        def forward(self, x_num: torch.Tensor) -> torch.Tensor:
            v = (2.0 * math.pi) * self.c.unsqueeze(0) * x_num.unsqueeze(-1)
            pe = torch.cat([torch.sin(v), torch.cos(v)], dim=-1)
            out = torch.einsum('bnk,nkd->bnd', pe, self.W) + self.b.unsqueeze(0)
            out = torch.relu(out)
            return out.reshape(out.shape[0], -1)


    class MLPPLR(nn.Module):
        """MLP-PLR classifier; forward returns a 1-D logit per instance."""

        def __init__(self, n_num_features: int, n_bin_features: int,
                     k_periodic: int, sigma_init: float, d_embedding: int,
                     n_layers: int, hidden_dim: int, dropout: float,
                     num_col_idx, bin_col_idx):
            super().__init__()
            self.n_num_features = n_num_features
            self.n_bin_features = n_bin_features
            self.register_buffer('num_col_idx_buf',
                                 torch.as_tensor(num_col_idx, dtype=torch.long))
            self.register_buffer('bin_col_idx_buf',
                                 torch.as_tensor(bin_col_idx, dtype=torch.long))
            self.plr = PLREmbedding(n_num_features, k_periodic, sigma_init, d_embedding)

            input_dim = n_num_features * d_embedding + n_bin_features
            layers = []
            in_dim = input_dim
            for _ in range(n_layers):
                layers.extend([nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
                in_dim = hidden_dim
            self.backbone = nn.Sequential(*layers)
            self.head     = nn.Linear(hidden_dim, 1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x_num = x.index_select(1, self.num_col_idx_buf)
            x_bin = x.index_select(1, self.bin_col_idx_buf)
            emb   = self.plr(x_num)
            z     = torch.cat([emb, x_bin], dim=-1)
            h     = self.backbone(z)
            return self.head(h).squeeze(-1)


    def load_mlp_plr_replica(bundle_path, device):
        bundle = joblib.load(bundle_path)
        m = MLPPLR(**bundle['arch_config'])
        m.load_state_dict(bundle['state_dict'])
        m.to(device).eval()
        return m, bundle['scaler']


    def make_mlp_predict_proba(model, replica_scaler, num_idx, device,
                               batch_size: int = 4096):
        """Return a LIME-compatible predict_proba callable.

        LIME calls predict_proba(X_np) where X_np is (n, n_features) numpy.
        We re-scale numeric cols with the replica scaler (pass-through binaries),
        push to GPU in batches, run forward, sigmoid, return (n, 2) [neg, pos].
        """
        def _predict(X_np: np.ndarray) -> np.ndarray:
            X_sc = X_np.astype(np.float32, copy=True)
            X_sc[:, num_idx] = replica_scaler.transform(X_np[:, num_idx])
            out = np.empty(X_sc.shape[0], dtype=np.float32)
            for start in range(0, X_sc.shape[0], batch_size):
                chunk = torch.from_numpy(X_sc[start:start + batch_size]).to(device)
                with torch.no_grad():
                    logit = model(chunk)
                    probs = torch.sigmoid(logit).cpu().numpy()
                out[start:start + batch_size] = probs
            return np.stack([1.0 - out, out], axis=1)
        return _predict


    print('MLP-PLR setup complete.')

PyTorch : 2.10.0+cu128
Device  : NVIDIA L4
MLP-PLR setup complete.


In [22]:
# ═════════════════════════════════════════════════════════════════════════════
# LIME helpers — explainer construction + attribution extraction
# ═════════════════════════════════════════════════════════════════════════════

def build_lime_explainer(training_data: np.ndarray,
                         feature_names: list,
                         bin_col_idx: list,
                         seed: int):
    """Build a LimeTabularExplainer for tabular binary classification.

    training_data : (n_ref, p) the sampling basis — typically the raw
                    window-A or window-B training matrix.
    bin_col_idx   : positions of binary (categorical) features; LIME samples these
                    from the empirical distribution instead of perturbing continuously.
    """
    return lime.lime_tabular.LimeTabularExplainer(
        training_data        = training_data,
        feature_names        = feature_names,
        categorical_features = bin_col_idx,   # binary cols treated as categorical
        class_names          = ['neg', 'pos'],
        discretize_continuous= False,
        sample_around_instance= True,
        mode                 = 'classification',
        random_state         = seed,
    )


def extract_lime_vector(lime_exp, n_features: int) -> np.ndarray:
    """Convert LIME's output into a full (n_features,) attribution vector.

    `as_map()` returns a dict { label: [(feature_idx, weight), ...] }. We extract
    the positive-class (label=1) weights; features not returned are set to 0.
    LIME returns weights for the top `num_features` by importance — we set
    num_features=n_features so the returned list covers all features.
    """
    out = np.zeros(n_features, dtype=np.float32)
    pos_label = 1
    for feat_idx, w in lime_exp.as_map()[pos_label]:
        out[feat_idx] = w
    return out


def build_subsample_idx(flagged_local_idx: np.ndarray,
                        p_hat_A: np.ndarray, p_hat_B: np.ndarray,
                        n_subsample: int, seed: int) -> np.ndarray:
    """Stratified-by-risk subsample of flagged_local_idx.

    Sort flagged instances by max(p_hat_A, p_hat_B) (which is how they were flagged),
    partition into n_subsample evenly-spaced deciles of rank, and sample one per decile.
    Returns positions *within flagged_local_idx*, i.e. values in range(len(flagged)).
    """
    rng = np.random.default_rng(seed)
    n_flagged = len(flagged_local_idx)
    if n_subsample >= n_flagged:
        return np.arange(n_flagged, dtype=np.int64)
    score = np.maximum(p_hat_A[flagged_local_idx], p_hat_B[flagged_local_idx])
    rank_order = np.argsort(-score)
    step = n_flagged / n_subsample
    picks = np.array([int(i * step + rng.uniform(0, step))
                      for i in range(n_subsample)], dtype=np.int64)
    picks = np.clip(picks, 0, n_flagged - 1)
    picks = np.unique(picks)
    if len(picks) < n_subsample:
        remaining = np.setdiff1d(np.arange(n_flagged), picks)
        extra = rng.choice(remaining, size=n_subsample - len(picks), replace=False)
        picks = np.concatenate([picks, extra])
    return np.sort(rank_order[picks[:n_subsample]]).astype(np.int64)


print('LIME helpers defined.')

LIME helpers defined.


In [23]:
# ═════════════════════════════════════════════════════════════════════════════
# Main LIME computation loop — one pass per window pair
#
# Per-window principle: A replicas are explained with a LIME explainer built
# from window-A training data (raw); B replicas with one built from window-B
# training data (raw). The flagged subsample is shared across A and B so
# attributions remain instance-wise comparable. All predict_proba callables
# receive raw input — each replica's own pipeline / scaler does scaling
# internally; LIME does not pre-scale.
# ═════════════════════════════════════════════════════════════════════════════

SEED_BASE = 42   # matches seeding convention in notebook 02 / 02b

if MODEL_TYPE not in ('xgboost', 'logreg', 'mlp_plr'):
    raise ValueError(f'Unknown MODEL_TYPE: {MODEL_TYPE}')


def get_predict_proba_for_replica(pair_dir, AB, r):
    """Return (predict_fn, cleanup_handle) for the requested replica.

    All callables consume RAW feature input (shape (n, p)). The replica's
    own pipeline / scaler handles internal scaling.
    """
    bundle_path = pair_dir / f'replicas_{AB}' / f'model_r{r}.joblib'

    if MODEL_TYPE == 'xgboost':
        model = joblib.load(bundle_path)            # XGBClassifier — trained on raw
        return model.predict_proba, None

    elif MODEL_TYPE == 'logreg':
        pipe = joblib.load(bundle_path)             # Pipeline(StandardScaler, LogisticRegression)
        return pipe.predict_proba, None             # pipeline scales internally

    elif MODEL_TYPE == 'mlp_plr':
        m, replica_scaler = load_mlp_plr_replica(bundle_path, DEVICE)
        predict = make_mlp_predict_proba(m, replica_scaler, num_col_idx, DEVICE)
        return predict, m

    raise ValueError(MODEL_TYPE)


for p in pairs:
    pid       = p['pair_id']
    pair_dir  = MODEL_DIR / f'pair_{pid:02d}'
    lime_pair = LIME_DIR  / f'pair_{pid:02d}'

    lime_A_path     = lime_pair / 'lime_A.npy'
    lime_B_path     = lime_pair / 'lime_B.npy'
    subsample_path  = lime_pair / 'lime_flagged_idx.npy'

    if lime_A_path.exists() and lime_B_path.exists() and subsample_path.exists():
        print(f'Pair {pid:02d}: LIME already computed, skipping.')
        continue

    print(f'\n── Pair {pid:02d} [{MODEL_TYPE}] ───────────────────────────────')
    lime_pair.mkdir(parents=True, exist_ok=True)

    pred_data = np.load(pair_dir / 'predictions.npz')
    flagged_local_idx = pred_data['flagged_idx']
    p_hat_A           = pred_data['p_hat_A']
    p_hat_B           = pred_data['p_hat_B']
    idx_A    = np.array(p['idx_A'],    dtype=np.int64)
    idx_B    = np.array(p['idx_B'],    dtype=np.int64)
    idx_eval = np.array(p['idx_eval'], dtype=np.int64)

    if len(flagged_local_idx) == 0:
        print('  WARNING: no flagged instances — skipping.')
        continue

    lime_sub = build_subsample_idx(
        flagged_local_idx, p_hat_A, p_hat_B,
        n_subsample=N_LIME_SUBSAMPLE,
        seed=SEED_BASE + pid * 100,
    )
    n_lime = len(lime_sub)
    print(f'  Flagged: {len(flagged_local_idx):,}  LIME subsample: {n_lime}')
    np.save(subsample_path, lime_sub)

    flagged_global_idx = idx_eval[flagged_local_idx]
    X_explain          = X[flagged_global_idx[lime_sub]]   # (n_lime, p) RAW
    n_feat             = X_explain.shape[1]

    X_basis_A = X[idx_A]
    X_basis_B = X[idx_B]

    explainer_A = build_lime_explainer(
        training_data = X_basis_A,
        feature_names = feature_names,
        bin_col_idx   = bin_col_idx,
        seed          = SEED_BASE + pid * 100 + 1,
    )
    explainer_B = build_lime_explainer(
        training_data = X_basis_B,
        feature_names = feature_names,
        bin_col_idx   = bin_col_idx,
        seed          = SEED_BASE + pid * 100 + 2,
    )
    print(f'  Per-window LIME explainers built: '
          f'A on {X_basis_A.shape[0]:,} rows, B on {X_basis_B.shape[0]:,} rows')

    lime_A = np.zeros((R, n_lime, n_feat), dtype=np.float32)
    lime_B = np.zeros((R, n_lime, n_feat), dtype=np.float32)

    for AB, base_explainer, out_arr in [
        ('A', explainer_A, lime_A),
        ('B', explainer_B, lime_B),
    ]:
        for r in range(R):
            predict_fn, cleanup_handle = get_predict_proba_for_replica(pair_dir, AB, r)

            for i in range(n_lime):
                exp = base_explainer.explain_instance(
                    data_row     = X_explain[i],
                    predict_fn   = predict_fn,
                    num_features = n_feat,
                    num_samples  = NUM_SAMPLES,
                )
                out_arr[r, i] = extract_lime_vector(exp, n_feat)

            mean_abs = float(np.abs(out_arr[r]).mean())
            print(f'  {AB} r{r:2d}: mean |LIME| = {mean_abs:.5f}')

            if MODEL_TYPE == 'mlp_plr' and cleanup_handle is not None:
                del cleanup_handle
                torch.cuda.empty_cache()

    np.save(lime_A_path, lime_A)
    np.save(lime_B_path, lime_B)
    print(f'  Saved lime_A {lime_A.shape}, lime_B {lime_B.shape}')

print('\n✓ All LIME attributions computed.')


── Pair 00 [mlp_plr] ───────────────────────────────
  Flagged: 7,629  LIME subsample: 200
  Per-window LIME explainers built: A on 172,623 rows, B on 195,649 rows
  A r 0: mean |LIME| = 0.00384
  A r 1: mean |LIME| = 0.00477
  B r 0: mean |LIME| = 0.00095
  B r 1: mean |LIME| = 0.00306
  Saved lime_A (2, 200, 683), lime_B (2, 200, 683)

── Pair 01 [mlp_plr] ───────────────────────────────
  Flagged: 9,465  LIME subsample: 200
  Per-window LIME explainers built: A on 195,649 rows, B on 216,541 rows
  A r 0: mean |LIME| = 0.00233
  A r 1: mean |LIME| = 0.00054
  B r 0: mean |LIME| = 0.00341
  B r 1: mean |LIME| = 0.00157
  Saved lime_A (2, 200, 683), lime_B (2, 200, 683)

── Pair 02 [mlp_plr] ───────────────────────────────
  Flagged: 6,375  LIME subsample: 200
  Per-window LIME explainers built: A on 216,541 rows, B on 248,283 rows
  A r 0: mean |LIME| = 0.00176
  A r 1: mean |LIME| = 0.00054
  B r 0: mean |LIME| = 0.00102
  B r 1: mean |LIME| = 0.00088
  Saved lime_A (2, 200, 683), l

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# LIME stochasticity diagnostic — run the best A-replica twice with two seeds
# on the same subsampled instances; compare. Uses the window-A training data
# (raw) as the LIME sampling basis, matching the per-window principle.
# ═════════════════════════════════════════════════════════════════════════════

for p in pairs:
    pid       = p['pair_id']
    pair_dir  = MODEL_DIR / f'pair_{pid:02d}'
    lime_pair = LIME_DIR  / f'pair_{pid:02d}'
    lime_pair.mkdir(parents=True, exist_ok=True)

    print(f'\n── Pair {pid:02d} [{MODEL_TYPE}] stochasticity diagnostic ──')

    pred_data = np.load(pair_dir / 'predictions.npz')
    idx_A    = np.array(p['idx_A'],    dtype=np.int64)
    idx_eval = np.array(p['idx_eval'], dtype=np.int64)
    flagged_local_idx = pred_data['flagged_idx']

    if len(flagged_local_idx) == 0:
        print('  No flagged instances — skipping diagnostic.')
        continue

    subsample_path = lime_pair / 'lime_flagged_idx.npy'
    if not subsample_path.exists():
        print('  lime_flagged_idx.npy missing — run main loop first. Skipping diagnostic.')
        continue
    lime_sub = np.load(subsample_path)
    n_lime   = len(lime_sub)

    flagged_global_idx = idx_eval[flagged_local_idx]
    X_explain          = X[flagged_global_idx[lime_sub]]   # raw
    X_basis_A          = X[idx_A]                          # raw

    n_feat = X_explain.shape[1]

    per_auc_A = pred_data['per_replica_pr_auc_A']
    best_r    = int(np.argmax(per_auc_A))
    print(f'  Best replica of A: r={best_r}  (PR-AUC={per_auc_A[best_r]:.4f})')

    predict_fn, cleanup_handle = get_predict_proba_for_replica(pair_dir, 'A', best_r)

    run1 = np.zeros((n_lime, n_feat), dtype=np.float32)
    run2 = np.zeros((n_lime, n_feat), dtype=np.float32)

    for seed, arr in [(0, run1), (1, run2)]:
        explainer = build_lime_explainer(
            training_data = X_basis_A,
            feature_names = feature_names,
            bin_col_idx   = bin_col_idx,
            seed          = seed,
        )
        for i in range(n_lime):
            exp = explainer.explain_instance(
                data_row     = X_explain[i],
                predict_fn   = predict_fn,
                num_features = n_feat,
                num_samples  = NUM_SAMPLES,
            )
            arr[i] = extract_lime_vector(exp, n_feat)

    max_abs_diff    = float(np.abs(run1 - run2).max())
    median_abs_diff = float(np.median(np.abs(run1 - run2)))
    is_det = max_abs_diff < 1e-6

    print(f'  Max    |run1 - run2|: {max_abs_diff:.4e}')
    print(f'  Median |run1 - run2|: {median_abs_diff:.4e}')
    print('  LIME is deterministic ✓' if is_det else '  LIME is stochastic (expected) ✓')

    stoch = {
        'best_replica_A':   best_r,
        'max_abs_diff':     max_abs_diff,
        'median_abs_diff':  median_abs_diff,
        'is_deterministic': is_det,
    }
    with open(lime_pair / 'stochasticity.json', 'w') as f:
        json.dump(stoch, f, indent=2)
    print(f'  Saved stochasticity.json')

    if MODEL_TYPE == 'mlp_plr' and cleanup_handle is not None:
        del cleanup_handle
        torch.cuda.empty_cache()

print('\n✓ Stochasticity diagnostic complete.')


── Pair 00 [mlp_plr] stochasticity diagnostic ──
  Best replica of A: r=1  (PR-AUC=0.1371)


In [ ]:
import matplotlib.pyplot as plt

lime_pair0 = LIME_DIR / 'pair_00' / 'lime_A.npy'
if lime_pair0.exists():
    lime_A_0 = np.load(lime_pair0)   # (R, n_lime, p)

    phi_bar = lime_A_0.mean(axis=0)                 # (n_lime, p)
    global_imp = np.abs(phi_bar).mean(axis=0)       # (p,)

    top_k = 20
    top_idx = np.argsort(global_imp)[::-1][:top_k]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(
        [feature_names[i] for i in top_idx[::-1]],
        global_imp[top_idx[::-1]],
        color='darkviolet'
    )
    ax.set_title(f'Top {top_k} global LIME importances (pair 0, model A, {MODEL_TYPE})')
    ax.set_xlabel('Mean |LIME weight|')
    plt.tight_layout()
    plt.savefig(LIME_DIR / 'global_importance_pair00_A.png', dpi=120)
    plt.show()
else:
    print(f'{lime_pair0} not found — skip summary plot.')